# Course 3 : Adaptive step size for Euler method 

In [ ]:
import numpy as np
from scipy.optimize import root
from scipy.integrate import solve_ivp
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.io as pio
pio.templates.default = "seaborn"

## Error estimator

We can construct an estimator of the local truncation error from the iterates of the explicit Euler method.
The local truncation error can be estimated from the second derivative of the solution :

$$ \mathcal{E}_{n} = \frac{\Delta t_{n}^2}{2} y''(t_{n}) + O(\Delta t_{n}^3).$$

When using an Euler-Cauchy or explicit Euler type solver, an estimator of the local truncation error can be given by :

$$
 \mathcal{E}_{n} = \frac{\Delta t_{n}^2}{2 \Delta t_{n-1}}  \left ( f(t_{n}, y_{n}) - f(t_{n-1}, y_{n-1})\right ) + O(\Delta t_{n}^3), \qquad y''(t_{n}) \simeq \frac{f(t_{n}, y_{n}) - f(t_{n-1}, y_{n-1})}{\Delta t_{n-1}}.
$$

If we want to adapt the time step in order to obtain a local truncation error increased by a fixed tolerance $\epsilon$, we can choose the time step at time $n$ by the formula : 

$$
\Delta t_{n} =   \sqrt{\frac{2\epsilon \Delta t_{n-1}}{ | f(t_{n}, y_n) - f(t_{n-1}, y_{n-1}) |} },
$$

as long as an initial time step is also provided. 

In [ ]:
#####################################################
class ode_result:
    def __init__(self, y, t):
        self.y = y
        self.t = t

#####################################################
def forward_euler(tini, tend, nt, yini, fcn):

    dt = (tend-tini) / (nt-1)
    t = np.linspace(tini, tend, nt)

    yini = np.atleast_1d(yini)
    neq = yini.size

    y = np.zeros((neq, nt))
    y[:,0] = yini

    for it, tn  in enumerate(t[:-1]):
        yn = y[:,it]
        y[:,it+1] = yn + dt*np.array(fcn(tn, yn))

    return ode_result(y, t)

#####################################################
def forward_euler_adapted(tini, tend, dtini, yini, fcn, tol=1e-6):

    t = [tini];

    yini = np.atleast_1d(yini)
    y = [yini]
        
    dt = dtini
    tn = tini
    it = 0
    
    while (tn<tend):
        yn = y[it]
        y.append(yn + dt*np.array(fcn(tn, yn)))
        tn += dt
        t.append(tn)
        ddy = abs(y[it+1]-yn)/dt
        dt = np.sqrt((2.0*tol)/np.linalg.norm(ddy))
        if (tn+dt)>tend: dt = tend-tn
        it += 1
        
    y = np.stack(y, axis=1)
    t = np.array(t)
    
    return ode_result(y, t)

## Curtiss and Hirschfelder problem

We consider the following problem :

$$
\left\{ 
\begin{aligned} 
{\mathrm d}_t u(t) & = k \, \big(cos(t) - u(t)) \big) \quad \text{with } k > 1\\ 
u(0)  & = u_0 
\end{aligned} 
\right. 
$$

In [ ]:
class curtiss_model:

    def __init__(self, k):
        self.k = k

    def fcn(self, t, u) :
        k = self.k
        u_dot = k * (np.cos(t) - u)
        return u_dot

    def sol(self, uini, t0, t):
        k = self.k

        c0 = (uini - (k/(k*k + 1)) * (k*np.cos(t0) + np.sin(t0))) * np.exp(k*t0)
        u = (k/(k*k + 1)) * (k*np.cos(t) + np.sin(t)) +  c0 * np.exp(-k*t)
        return u

### Forward Euler with adaptive time step

In [ ]:
def plot_fe_sol(k, tini, tend, nt, uini, tol):
    cm = curtiss_model(k)
    fcn = cm.fcn
    
    fig =  make_subplots(rows=2, cols=1, vertical_spacing=0.1, subplot_titles=("Solution", "Error"))
    
    sol = forward_euler(tini, tend, nt, uini, fcn)
    err = np.abs(sol.y[0] - cm.sol(uini, tini, sol.t))
    
    fig.add_trace(go.Scatter(x=sol.t, y=sol.y[0], name=f'fixed dt, nt={sol.t.size}', 
                             line_color='rgb(76,114,176)', mode='markers+lines', legendgroup='1'), row=1, col=1)
    fig.add_trace(go.Scatter(x=sol.t, y=err, name=f'fixed dt, nt={sol.t.size}', 
                             line_color='rgb(76,114,176)', mode='markers+lines', legendgroup='2'), row=2, col=1)
    
    dtini = 1e-4
    nt_adapt = np.empty(len(tol), dtype=int)
    for i, tol_i in enumerate(tol):
        sol_adapt = forward_euler_adapted(tini, tend, dtini, uini, fcn, tol=tol_i)
        err_adapt = np.abs(sol_adapt.y[0] - cm.sol(uini, tini, sol_adapt.t))
        nt_adapt[i] = sol_adapt.t.size
        fig.add_trace(go.Scatter(visible=False, x=sol_adapt.t, y=sol_adapt.y[0], name=f'adapted dt, nt={sol_adapt.t.size}', 
                                 line_color='rgb(221,132,82)', mode='markers+lines', legendgroup='1'), row=1, col=1)
        fig.add_trace(go.Scatter(visible=False, x=sol_adapt.t, y=err_adapt, name=f'adapted dt, nt={sol_adapt.t.size}', 
                                 line_color='rgb(221,132,82)', mode='markers+lines', legendgroup='2'), row=2, col=1)
    
    fig.data[-2].visible = True
    fig.data[-1].visible = True
    
    # Create and add slider
    steps = []
    for i, tol_i in enumerate(tol):
        args = [{"visible": [(el==0) or (el==1) or (el==2*i+2) or (el==2*i+3) for el in range(len(fig.data))]}]
        step = dict(method="update", label = f"{tol_i:.2e}", args=args)
        steps.append(step)
    sliders = [dict(active=len(tol)-1, currentvalue={'prefix': 'Tolerance = '}, steps=steps)]

    #print("Explicit Euler with fixed time step:", end="")
    #print(f"  nb of time points = {nt}")   
    for i, tol_i in enumerate(tol):
        print(f"Forward Euler with adaptive time step tol = {tol_i:.2e}:", end="")
        print(f"  nb of time points = {nt_adapt[i]}")

    legend=dict(x=0.70,y=0.98, bgcolor='rgba(0,0,0,0)')
    fig.update_layout(sliders=sliders, height=800, legend=legend, legend_tracegroupgap=300, legend_groupclick="toggleitem")
    fig.show()

In [ ]:
k = 100.

uini = (2.,)

tini = 0.
tend = 0.5
nt = 400

k = 100.

tol = np.array([1e-4, 1e-5, 1e-6])

plot_fe_sol(k, tini, tend, nt, uini, tol)

This method can be experimented with several tolerances. Note that choosing a tolerance of $10^{-5}$ leads to a solution with an error lower than a solution with $400$ points, but with a number of points about four times smaller. Choosing a larger tolerance of 10-4 leads to behavior with significant errors outside the initial time boundary layer. Explain this phenomenon.

**Construction of the global error from the local error**

Let us recall that the equation we want to resolve in the Dahlquist model equation in the linear framework:

$$
\begin{cases}
    \mathrm{d}_{\tau} \eta_1 &= \lambda T \,\eta_1, \qquad \tau \in [0, 1], \tau_0 = \frac{1}{\lambda T}\\
    \eta_1(\tau = 0) &=  \overline{y}_{10},\\
\end{cases}
$$    

with the forward Euler scheme:      

$$     
\begin{cases}
    y_{1, n+1} &= y_{1, n} + \lambda T \Delta \tau_n y_{1, n} = (1 + \lambda T \Delta \tau_n) y_{1, n} =  \left(1 - \frac{\Delta \tau_n}{\tau_0}\right)^n y_{10}, 
\end{cases}
$$

Estimating the global error in this context is not too hard:

$$
\begin{cases}
    e_{1, n+1} &= e_{1, n} + \lambda T \Delta \tau_n e_{1, n} + \varepsilon_{1, n} =  \left(1 - \frac{\Delta \tau_n}{\tau_0}\right) e_{1, n} + \varepsilon_{1, n}\\
    e_{1, n} & = \sum_{l=0}^{n-1}\left(\prod_{k=l+1}^{n-1}\left(1 - \frac{\Delta \tau_k}{\tau_0}\right)\varepsilon_{1,l}\right).
\end{cases}
$$

Within the initial temporal boundary layer, the second derivative of the solution scales like $1/\tau_0^2$, which implies a small time step with respect to the boundary layer thickness $\tau_0$; however outside the boundary layer, on the slow manifold, the factor  $ \left(1 - \frac{\Delta \tau_n}{\tau_0}\right)$ becomes large and even negative with a high amplitude so that the global error is getting strongly amplified, whereas the local error remains small in size.

The local estimator is not sufficient since it is only based on the local error estimate and can yield time steps that are out of the sstability limit, thus lmacking robustness.

### Backward Euler with adaptive time step

In [ ]:
#####################################################
def backward_euler(tini, tend, nt, yini, fcn):

    dt = (tend-tini) / (nt-1)
    t = np.linspace(tini, tend, nt)

    yini = np.atleast_1d(yini)
    neq = yini.size

    y = np.zeros((neq, nt))
    y[:,0] = yini

    def g(uip1, *args):
        uip, tip1 = args
        return uip1 - uip - dt*np.array(fcn(tip1, uip1))

    for it, tn  in enumerate(t[:-1]):
        yn = y[:,it]
        y0 = yn + dt*np.array(fcn(tn, yn))
        # solve y[:,it+1] - y[:,it] - dt * fcn(tini + (it+1)*dt, y[:,it+1]) = 0
        sol = root(g, y0, (yn, tn+dt))
        y[:,it+1] = sol.x

    return ode_result(y, t)

#####################################################
def backward_euler_adapted(tini, tend, dtini, yini, fcn, tol=1e-6):
    
    t = [tini];

    yini = np.atleast_1d(yini)
    y = [yini]
    
    dt = dtini
    tn = tini
    it = 0

    def g(uip1, *args):
        uip, tip1 = args
        return uip1 - uip - dt*np.array(fcn(tip1, uip1))
    
    
    while (tn<tend):
        yn = y[it]
        y0 = yn + dt*np.array(fcn(tn, yn))
        # solve y[:,it+1] - y[:,it] - dt * fcn(tini + (it+1)*dt, y[:,it+1]) = 0
        sol = root(g, y0, (yn, tn+dt))
        y.append(sol.x)
        tn += dt
        t.append(tn)
        ddy = abs(y[it+1]-yn)/dt
        dt = np.sqrt(2*tol/np.linalg.norm(ddy))
        if (tn+dt)>tend: dt = tend-tn
        it += 1
        
    y = np.stack(y, axis=1)
    t = np.array(t)

    return ode_result(y, t)

#####################################################
def plot_be_sol(k, tini, tend, nt, uini, tol):
    cm = curtiss_model(k)
    fcn = cm.fcn
    
    fig = make_subplots(rows=2, cols=1, subplot_titles=("Solution", "Error"))
    
    sol = backward_euler(tini, tend, nt, uini, fcn)
    err = np.abs(sol.y[0] - cm.sol(uini, tini, sol.t))
    
    color = fig.layout.template.layout.colorway
    fig.add_trace(go.Scatter(x=sol.t, y=sol.y[0], name=f'fixed dt, nt={sol.t.size}', mode='markers+lines', line_color=color[0], legendgroup='sol'), row=1, col=1)
    fig.add_trace(go.Scatter(x=sol.t, y=err, name=f'fixed dt, nt={sol.t.size}', mode='markers+lines', line_color=color[0], legendgroup='err'), row=2, col=1)
    
    dtini = 1e-4
    nt_adapt = np.empty(len(tol), dtype=int)
    for i, tol_i in enumerate(tol):
        sol_adapt = backward_euler_adapted(tini, tend, dtini, uini, fcn, tol=tol_i)
        err_adapt = np.abs(sol_adapt.y[0] - cm.sol(uini, tini, sol_adapt.t))
        nt_adapt[i] = sol_adapt.t.size
        fig.add_trace(go.Scatter(visible=False, x=sol_adapt.t, y=sol_adapt.y[0], name=f'adapted dt, nt={sol_adapt.t.size}', 
                                 line_color=color[1], mode='markers+lines', legendgroup='sol'), row=1, col=1)
        fig.add_trace(go.Scatter(visible=False, x=sol_adapt.t, y=err_adapt, name=f'adapted dt, nt={sol_adapt.t.size}', 
                                 line_color=color[1], mode='markers+lines', legendgroup='err'), row=2, col=1)

    fig.data[2].visible = True
    fig.data[3].visible = True
    
    steps = []
    for i, tol_i in enumerate(tol):
        args = [{"visible": [(el==0) or (el==1) or (el==2*i+2) or (el==2*i+3) for el in range(len(fig.data))]}]
        step = dict(method="update", label=f"{tol_i:.2e}", args=args)
        steps.append(step)
    sliders = [dict(active=len(tol)-1, currentvalue={'prefix': 'tol = '}, steps=steps)]

    for i, tol_i in enumerate(tol):
        print(f"Backward Euler with adaptive time step tol = {tol_i:.2e}:", end="")
        print(f"  nb of time points = {nt_adapt[i]}")
    
    legend = dict(x=0.8, bgcolor='rgba(0,0,0,0)', tracegroupgap=300, groupclick="toggleitem")
    fig.update_xaxes(title='t')
    fig.update_layout(height=750, sliders=sliders, legend=legend)
    fig.show()

In [ ]:
k = 100.

uini = (2.,)

tini = 0.
tend = 0.5
nt = 400

tol = [1e-3, 1e-4, 1e-5, 1e-6]

plot_be_sol(k, tini, tend, nt, uini, tol)

In the case of implicit Euler, this question does not arise for a dissipative differential equation of the Hirschfelder and Curtis type, and good results are obtained, even for small tolerances. On the other hand, this is not necessarily the case for the explosive problems (positive eigenvalue of large amplitude) studied below.

## Thermal explosion test

In [ ]:
class explosion_with_consumption_1eq_model:

    def __init__(self, Tr, beta):
        self.Tr = Tr
        self.beta = beta 

    def fcn(self, t, theta):
        beta = self.beta
        Tr = self.Tr
        theta_dot = np.exp(theta/(1+(theta/beta))) * (1-theta/Tr) 
        return theta_dot
    
    def jac(self, t, theta):
        beta = self.beta
        Tr = self.Tr
        exp = np.exp(theta/(1+(theta/beta)))
        return (exp/(1+theta/beta)**2) * (1-theta/Tr) - exp/Tr

### Forward Euler

In [ ]:
def exp_plot_fe_sol(Tr, beta, tini, tend, nt, yini, tol):
    emwc1eq = explosion_with_consumption_1eq_model(Tr, beta)
    fcn = emwc1eq.fcn
    jac = emwc1eq.jac
    
    fig = make_subplots(rows=2, cols=1, subplot_titles=("Solution", "Error"))
    
    sol_ref = solve_ivp(fcn, (tini, tend), yini, method='Radau', atol=1.e-8, rtol=1.e-8)
    
    nt = 15001
    sol = forward_euler(tini, tend, nt, yini, fcn)
    sol_tmp = solve_ivp(fcn, (tini, tend), yini, method='Radau', t_eval=sol.t, atol=1.e-8, rtol=1.e-8)
    err = np.abs(sol.y[0] - sol_tmp.y[0])
    ##print("Explicit Euler :", end="")
    ##print(f"  nb of time points = {nt}")
    
    color = fig.layout.template.layout.colorway
    fig.add_trace(go.Scattergl(x=sol_ref.t, y=sol_ref.y[0], name='sol_ref', mode='markers+lines', line_color=color[7], legendgroup='sol'), row=1, col=1)
    fig.add_trace(go.Scattergl(x=sol.t, y=sol.y[0], name=f'fixed dt, nt={sol.t.size}', mode='markers+lines', line_color=color[0], legendgroup='sol'), row=1, col=1)
    fig.add_trace(go.Scattergl(x=sol.t, y=err, name=f'fixed dt, nt={sol.t.size}', mode='markers+lines', line_color=color[0], legendgroup='err'), row=2, col=1)
    
    dtini = 1e-4
    nt_adapt = np.empty(len(tol), dtype=int)
    for i, tol_i in enumerate(tol):
        sol_adapt = forward_euler_adapted(tini, tend, dtini, yini, fcn, tol=tol_i)
        sol_tmp = solve_ivp(fcn, (tini, tend), yini, method='Radau', t_eval=sol_adapt.t, atol=1.e-8, rtol=1.e-8)
        err_adapt = np.abs(sol_adapt.y[0] - sol_tmp.y[0])
        ##print(f"Adaptive explicit Euler with a tolerance of {tol_i:.2e}:", end="")
        ##print(f"  nb of time points = {sol_adapt.t.size}")
        nt_adapt[i] = sol_adapt.t.size
        fig.add_trace(go.Scatter(visible=False, x=sol_adapt.t, y=sol_adapt.y[0], name=f'adapted dt, nt={sol_adapt.t.size}', 
                                 line_color=color[1], mode='markers+lines', legendgroup='sol'), row=1, col=1)
        fig.add_trace(go.Scatter(visible=False, x=sol_adapt.t, y=err_adapt, name=f'adapted dt, nt={sol_adapt.t.size}', 
                                 line_color=color[1], mode='markers+lines', legendgroup='err'), row=2, col=1)
    
    fig.data[-1].visible = True
    fig.data[-2].visible = True

    steps = []
    for i, tol_i in enumerate(tol):
        args = [{"visible": [(el==0) or (el==1) or (el==2) or (el==2*i+3) or (el==2*i+4) for el in range(len(fig.data))]}]
        step = dict(method="update", label=f"{tol_i:.2e}", args=args)
        steps.append(step)
    sliders = [dict(active=len(tol)-1, currentvalue={ 'prefix': 'tol = '}, steps=steps)]

    #print("Explicit Euler :", end="")
    #print(f"  nb of time points = {nt}")   
    for i, tol_i in enumerate(tol):
        print(f"Forward Euler with adaptive time step tol = {tol_i:.2e}:", end="")
        print(f"  nb of time points = {nt_adapt[i]}")

    legend = dict(x=0.8, y=0.6, bgcolor='rgba(0,0,0,0)', tracegroupgap=250, groupclick="toggleitem")
    fig.update_xaxes(title='t')
    fig.update_layout(height=750, sliders=sliders, legend=legend)
    fig.show()

In [ ]:
Tr = 200.0
beta = 15. 

tini = 0.0
tend = 5.0
nt = 15001

yini = (0.,)

tol =[1e-3, 1e-4, 1e-5, 1e-6]

exp_plot_fe_sol(Tr, beta, tini, tend, nt, yini, tol)

In this case, the difficulties encountered with the previous dissipative case are repeated within the area close to the final equilibrium point, where the very high stiffness of the system causes problems, even at small tolerances.

### Backward Euler

In [ ]:
def exp_plot_be_sol(Tr, beta, tini, tend, nt, yini, tol):
    emwc1eq = explosion_with_consumption_1eq_model(Tr, beta)
    fcn = emwc1eq.fcn
    jac = emwc1eq.jac
    
    fig = make_subplots(rows=2, cols=1, subplot_titles=("Solution", "Error"))
    
    sol_ref = solve_ivp(fcn, (tini, tend), yini, method='Radau', atol=1.e-8, rtol=1.e-8)
    
    sol = backward_euler(tini, tend, nt, yini, fcn)
    sol_tmp = solve_ivp(fcn, (tini, tend), yini, method='Radau', t_eval=sol.t, atol=1.e-8, rtol=1.e-8)
    err = np.abs(sol.y[0] - sol_tmp.y[0])
    #print("Implicit Euler :", end="")
    #print(f"  nb of time points = {nt}")
    
    color = fig.layout.template.layout.colorway
    fig.add_trace(go.Scattergl(x=sol_ref.t, y=sol_ref.y[0], name='sol_ref', mode='markers+lines', line_color=color[7], legendgroup='sol'), row=1, col=1)
    fig.add_trace(go.Scattergl(x=sol.t, y=sol.y[0], name=f'fixed dt, nt={sol.t.size}', mode='markers+lines', line_color=color[0], legendgroup='sol'), row=1, col=1)
    fig.add_trace(go.Scattergl(x=sol.t, y=err, name=f'fixed dt, nt={sol.t.size}', mode='markers+lines', line_color=color[0], legendgroup='err'), row=2, col=1)
    
    dtini = 1e-4
    nt_adapt = np.empty(len(tol), dtype=int)
    for i, tol_i in enumerate(tol):
        sol_adapt = backward_euler_adapted(tini, tend, dtini, yini, fcn, tol=tol_i)
        sol_tmp = solve_ivp(fcn, (tini, tend), yini, method='Radau', t_eval=sol_adapt.t, atol=1.e-8, rtol=1.e-8)
        err_adapt = np.abs(sol_adapt.y[0] - sol_tmp.y[0])
        #print(f"Adaptive implicit Euler with a tolerance of {tol_i:.2e}:", end="")
        #print(f"  nb of time points = {sol_adapt.t.size}")
        nt_adapt[i] = sol_adapt.t.size
        fig.add_trace(go.Scattergl(visible=False, x=sol_adapt.t, y=sol_adapt.y[0],  name=f'adapted dt, nt={sol_adapt.t.size}', 
                                 line_color=color[1], mode='markers+lines', legendgroup='sol'), row=1, col=1)
        fig.add_trace(go.Scattergl(visible=False, x=sol_adapt.t, y=err_adapt,  name=f'adapted dt, nt={sol_adapt.t.size}', 
                                 line_color=color[1], mode='markers+lines', legendgroup='err'), row=2, col=1)
    
    fig.data[-1].visible = True
    fig.data[-2].visible = True
    
    steps = []
    for i, tol_i in enumerate(tol):
        args = [{"visible": [(el==0) or (el==1) or (el==2) or (el==2*i+3) or (el==2*i+4) for el in range(len(fig.data))]}]
        step = dict(method="update", label=f"{tol_i:.2e}", args=args)
        steps.append(step)
    sliders = [dict(active=len(tol)-1, currentvalue={'prefix': 'tol = '}, steps=steps)]

    for i, tol_i in enumerate(tol):
        print(f"Backward Euler with adaptive time step tol = {tol_i:.2e}:", end="")
        print(f"  nb of time points = {nt_adapt[i]}")

        
    legend = dict(x=0.8, y=0.6, bgcolor='rgba(0,0,0,0)', tracegroupgap=230, groupclick="toggleitem")
    fig.update_xaxes(title='t')
    fig.update_layout(height=750, sliders=sliders, legend=legend)
    fig.show()

In [ ]:
Tr = 200.0
beta = 15. 

tini = 0.0
tend = 5.0
nt = 15001

yini = (0.,)

tol = [1.e-2, 1e-3, 1e-4, 1e-5, 1e-6]

exp_plot_be_sol(Tr, beta, tini, tend, nt, yini, tol)

In this explosive case, we encounter the same problems as in the case of fixed time step for implicit Euler. If the tolerance is not small enough, the explosion cannot be captured. This is the symmetric case compared to the steep case for explicit Euler. Here again, time-step adaptation does not provide a robust solution from this local error estimator, which works relatively well in general and is inexpensive but lacks robustness. A more robust adaptation strategy will be presented in the next chapter.